<a href="https://colab.research.google.com/github/rselvak6/opendde-colab/blob/main/opendde.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OpenDDE on Colab

Requires a GPU runtime: **Runtime > Change runtime type > T4 GPU** (or better).

This notebook: installs OpenDDE from source with `uv`, downloads model weights + inference data, runs a tiny smoke test, then runs your real prediction job.

In [1]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
!test -d OpenDDE || git clone https://github.com/aurekaresearch/OpenDDE.git
%cd OpenDDE
!uv pip install --torch-backend cu126 -e '.[gpu]'
!uv pip install --group dev
!opendde doctor

downloading uv 0.11.29 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
Cloning into 'OpenDDE'...
remote: Enumerating objects: 470, done.
remote: Counting objects: 100% (470/470), done.
remote: Compressing objects: 100% (285/285), done.
remote: Total 470 (delta 186), reused 443 (delta 167), pack-reused 0 (from 0)
Receiving objects: 100% (470/470), 20.68 MiB | 29.13 MiB/s, done.
Resolving deltas: 100% (186/186), done.
/content/OpenDDE
Using Python 3.12.13 environment at: /usr
Resolved 97 packages in 1.64s
Prepared 46 packages in 48.20s
Uninstalled 28 packages in 863ms
Installed 46 packages in 209ms
 + biopython==1.85
 + biotite==1.4.0
 + biotraj==1.2.2
 + comm==0.2.3
 + cuequivariance==0.10.0
 + cuequivariance-ops-cu12==0.8.0
 + cuequivariance-ops-torch-cu12==0.8.0
 + cuequivariance-torch==0.8.0
 + gemmi==0.6.7
 + ihm==2.11
 - ipywidgets==7.7.1
 + ipywidgets==8.1.7
 + jedi==0.20.0
 - matplotlib==3.10.0
 + matplotlib==3.10.5
 + ml-collections==1.1.

In [2]:
# General-purpose checkpoint used by default with -n opendde_v1.
%env OPENDDE_ROOT_DIR=/content/data
!mkdir -p $OPENDDE_ROOT_DIR/checkpoint

env: OPENDDE_ROOT_DIR=/content/data


In [3]:
# The download script needs the `zstd` CLI (not the Python `zstd` package) to
# decompress the search-database .zst files, or it silently falls back to a
# broken S3 mirror (403).
!apt-get -qq update && apt-get -qq install -y zstd
!bash /content/OpenDDE/scripts/download_opendde_data.sh
# If you don't need MSA/template/RNA-MSA features (as in the pred command below),
# add --skip-search-database to skip several GB of files you won't use:
# !bash /content/OpenDDE/scripts/download_opendde_data.sh --skip-search-database

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
[INFO] Using OPENDDE_ROOT_DIR: /content/data
[INFO] Downloading common inference data files from https://huggingface.co/aurekaresearch/OpenDDE/resolve/main/common
[INFO] Downloading https://huggingface.co/aurekaresearch/OpenDDE/resolve/main/common/components.cif
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1043  100  1043    0     0   5938      0 --:--:-- --:--:-- --:--:--  5960
100

In [ ]:
# Smoke test with the minimal example from the OpenDDE README, to confirm
# the install + data download actually work before running a real job.
tiny_json = '''[
    {
        "name": "tiny",
        "modelSeeds": [101],
        "sequences": [
            {
                "proteinChain": {
                    "sequence": "ACDEFGHIK",
                    "count": 1
                }
            }
        ]
    }
]'''
with open("tiny.json", "w") as f:
    f.write(tiny_json)

!opendde pred -i tiny.json -o ./output -n opendde_v1 --use_msa false --use_template false --use_rna_msa false --sample 1 --step 200 --cycle 10

In [4]:
# Your real job. Upload/create egfr-rad51-kinase.json in this working directory
# first (it wasn't part of the original notebook, so it must exist before running).
!opendde pred -i egfr-rad51-kinase.json -o ./output -n opendde_v1 --use_msa false --use_template false --use_rna_msa false --sample 1 --step 200 --cycle 10 --need_atom_confidence true

2026-07-16 02:05:26,974 [/usr/local/lib/python3.12/dist-packages/numexpr/utils.py:164] INFO numexpr.utils: NumExpr defaulting to 12 threads.
2026-07-16 02:05:28,360 [/content/OpenDDE/runner/batch_inference.py:909] INFO runner.batch_inference: Run infer with input=egfr-rad51-kinase.json, out_dir=./output, sample=1
2026-07-16 02:05:28,360 [/content/OpenDDE/runner/batch_inference.py:918] INFO runner.batch_inference: Using inference params for model opendde_v1: cycle=10, step=200, use_msa=False
2026-07-16 02:05:28,360 [/content/OpenDDE/runner/batch_inference.py:512] INFO runner.batch_inference: Will infer with 1 jsons
2026-07-16 02:05:28,547 [/content/OpenDDE/opendde/config/inference.py:114] INFO opendde.config.inference: Resolved triangle kernels from auto to multiplicative=cuequivariance, attention=cuequivariance.
2026-07-16 02:05:28,548 [/content/OpenDDE/runner/batch_inference.py:396] INFO runner.batch_inference: Inference by OpenDDE: model_name: opendde_v1, dtype: fp32
2026-07-16 02:05

In [5]:
import shutil
from google.colab import files

shutil.make_archive("opendde_output", "zip", "./output")
files.download("opendde_output.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>